#Title: GLiNER2 for Performing Multiple NLP Tasks Efficiently

#### Member Names :

Lok Prakash Pandey, Samuel Ukoha, Xinquan Yu

#### Email :

[ lok.pandey@torontomu.ca, samuel.ukoha@toronmu.ca, xinquan.yu@torontomu.ca ]

# Introduction

#### Problem Description:
NLP tasks such as Named Entity Recognition (NER), Text Classification (TC), Structured Data Extraction (SDE) and Relation Extraction (RE) requires separate LLM models for each task. This increases engineering complexity and cost of the tasks associated.

#### Context of the Problem:
Performing NLP tasks is important in the real-world because they help with various tasks like understanding documents, classification of documents, data analysis and creating knowledge graphs. A practical NLP solution should be efficient, flexible, and easy to deploy on a limited hardware.

#### Limitations of Other Approaches:
- Traditional LLMs do exist to solve the problems of NER, TC, SDE and RE, but they require expensive GPUs [1].
- Task-based models, which means requiring task specific models to work on different NLP tasks, adds complexity to the NLP process which can be decreased by having one generic model [2].

#### Solution:
This project studies **GLiNER2** (Generalist and Lightweight model for Named Entity Recognition 2), which is a unified system for performing NER, TC, SDE and RE tasks. The implementation provided below reproduces its main usage pattern in Google Colab using the open-source package and pretrained model.

#Background

The following table summarizes the works done relevant to this project:

<table>
<tr>
<th>Reference</th>
<th>Explanation</th>
<th>Dataset/Input</th>
<th>Weakness</th>
</tr>

<tr>
<td>Zaratiana et al. [1]</td>
<td>
They proposed <b>GLiNER</b>, a generalist NER model using a bidirectional transformer.<br><br>
It leverages label descriptions to recognize both seen and unseen entity types without retraining.
</td>
<td>
Standard NER datasets with:<br>
• Text input<br>
• Label descriptions
</td>
<td>
Limited to NER tasks only
</td>
</tr>

<tr>
<td>Zaratiana et al. [2]</td>
<td>
They proposed <b>GLiNER2</b>, an extension of GLiNER.<br><br>
It supports multi-task information extraction using a schema-driven interface for entities and relations.
</td>
<td>
Multi-task IE datasets with:<br>
• Schema definitions (entities, relations)<br>
• Text input
</td>
<td>
Depends on schema design quality <br> and requires careful schema definition
</td>
</tr>

</table>

# Methodology

In this project, we implemented a simplified schema-driven information extraction method based on GLiNER2. We used the released pretrained model in Google Colab rather than training the model from scratch. The main idea of the method is that a single encoder-based model can perform different NLP tasks when given an appropriate schema, such as entity labels, classification labels, relation types, or structured fields.

Our implementation followed one unified workflow. First, we loaded the pretrained GLiNER2 model. Next, we defined schemas depending on the information we wanted to extract. Then, we ran the same model on input text and examined the outputs. Using this process, we tested the framework on classification, named entity recognition, relation extraction, and structured extraction examples. Although the outputs were different, they all came from the same schema-driven extraction method. This allowed us to study the central contribution of the paper in a manageable and practical way.

In [ ]:
!rm -rf /content/gliner2-base-v1

In [ ]:
!pip install -q gliner2
!apt-get -qq update
!apt-get -qq install git-lfs
!git lfs install

In [ ]:
!git clone https://huggingface.co/fastino/gliner2-base-v1 /content/gliner2-base-v1

In [ ]:
!ls -lh /content/gliner2-base-v1

In [ ]:
from gliner2 import GLiNER2

# This line loads the pretrained GLiNER2 model from the local folder.
# "extractor" is the main object we will use for all later tasks.
# We use the same model for classification, NER, relation extraction,
# and structured extraction, which matches the paper's unified framework idea.
extractor = GLiNER2.from_pretrained("/content/gliner2-base-v1")

# This print statement is only used to confirm that the model was loaded successfully.
print("Model loaded successfully.")

# 1. Text Classification

In [ ]:
# This section demonstrates text classification.
# The goal is to let the model assign one sentiment label
# to the input sentence.

# "schema" defines the task we want the model to perform.
# Here, we tell GLiNER2 that the task name is "sentiment".
# The candidate labels are "positive", "negative", and "neutral".
# Because we do not set multi_label=True, the model will choose only one label.
schema = extractor.create_schema().classification(
    "sentiment",
    ["positive", "negative", "neutral"]
)

# "text" is the input sentence that we want to classify.
# This sentence expresses a strong positive opinion.
text = "This product exceeded my expectations! Absolutely love it."

# extractor.extract(...) applies the schema to the input text.
# Since the schema is a classification schema, the model will return
# one sentiment label for this sentence.
results = extractor.extract(text, schema)

# Print the final output.
# We expect something similar to: {'sentiment': 'positive'}
print(results)

In [ ]:
# =========================
# Task 1: Text Classification on a real dataset (SST-2)
# Dataset: glue, sst2
# =========================

# This line installs the Hugging Face datasets library.
# We need it because SST-2 is loaded from Hugging Face.
!pip install -q datasets

# Import the dataset loader from Hugging Face.
from datasets import load_dataset

# Import GLiNER2.
from gliner2 import GLiNER2

# -------------------------
# Step 1: Load the pretrained GLiNER2 model
# -------------------------
# "extractor" is the main model object.
# We use the local model path that we already downloaded before.
extractor = GLiNER2.from_pretrained("/content/gliner2-base-v1")

# Print a message so we know the model is ready.
print("GLiNER2 model loaded successfully.")

# -------------------------
# Step 2: Load the SST-2 dataset
# -------------------------
# SST-2 is a sentiment classification dataset.
# Each example contains:
# - "sentence": the input text
# - "label": the ground-truth sentiment
#     0 = negative
#     1 = positive
dataset = load_dataset("glue", "sst2")

# We use the validation split instead of the training split.
# This is more suitable for a quick evaluation/demo.
validation_data = dataset["validation"]

# -------------------------
# Step 3: Select a small subset
# -------------------------
# To keep the notebook lightweight and fast, we only evaluate a small subset.
# You can change this number if needed.
sample_size = 50

# Select the first "sample_size" examples from the validation set.
sample_data = validation_data.select(range(sample_size))

# Print the subset size to confirm.
print(f"Number of evaluation samples: {len(sample_data)}")

# -------------------------
# Step 4: Define label mapping
# -------------------------
# GLiNER2 will output text labels such as "positive" or "negative".
# SST-2 stores labels as integers:
# 0 = negative
# 1 = positive
id_to_label = {
    0: "negative",
    1: "positive"
}

# -------------------------
# Step 5: Run classification on each sample
# -------------------------
# We will store:
# - gold_labels: the true labels from SST-2
# - pred_labels: the labels predicted by GLiNER2
# - detailed_results: sample-by-sample outputs for inspection/presentation
gold_labels = []
pred_labels = []
detailed_results = []

# Loop through each example in the selected subset.
for example in sample_data:
    # "sentence" is the input text.
    sentence = example["sentence"]

    # "label" is the true SST-2 label (0 or 1).
    gold_id = example["label"]

    # Convert the numeric gold label into text label.
    gold_label = id_to_label[gold_id]

    # Use GLiNER2 quick classification API.
    # We define one task called "sentiment" with two possible labels:
    # "positive" and "negative".
    prediction = extractor.classify_text(
        sentence,
        {"sentiment": ["positive", "negative"]}
    )

    # Extract the predicted sentiment label from the model output.
    pred_label = prediction["sentiment"]

    # Save labels for accuracy calculation.
    gold_labels.append(gold_label)
    pred_labels.append(pred_label)

    # Save detailed information for later printing.
    detailed_results.append({
        "sentence": sentence,
        "gold": gold_label,
        "pred": pred_label
    })

# -------------------------
# Step 6: Calculate accuracy
# -------------------------
# Accuracy = number of correct predictions / total number of predictions
correct = 0

# Compare each predicted label with the true label.
for gold, pred in zip(gold_labels, pred_labels):
    if gold == pred:
        correct += 1

accuracy = correct / len(gold_labels)

# Print the final performance result.
print(f"\nClassification Accuracy on SST-2 subset: {accuracy:.4f}")
print(f"Correct predictions: {correct}/{len(gold_labels)}")

# -------------------------
# Step 7: Print a few sample outputs
# -------------------------
# These examples are useful for presentation because they show
# what the input looks like and how the model predicted it.
print("\nSample predictions:")
for i in range(5):
    print(f"\nExample {i+1}")
    print("Sentence:", detailed_results[i]["sentence"])
    print("Gold label:", detailed_results[i]["gold"])
    print("Predicted label:", detailed_results[i]["pred"])

# 2. Named Entity Recognition (NER)

In [ ]:
# This section demonstrates named entity recognition (NER).
# The goal is to identify important entities in the sentence,
# such as company names, people, products, locations, and dates.

# This schema tells the model which entity types to extract.
# The model will scan the text and try to find spans that belong
# to these five categories.
schema = extractor.create_schema().entities([
    "company", "person", "product", "location", "date"
])

# This is the input sentence for NER.
# It contains multiple entity types:
# - Apple Inc. -> company
# - Tim Cook -> person
# - iPhone 15 -> product
# - Cupertino, California -> location
# - September 12, 2023 -> date
text = "Apple Inc. CEO Tim Cook announced the new iPhone 15 in Cupertino, California on September 12, 2023."

# extractor.extract(...) now performs entity extraction because the schema
# is defined with .entities(...).
# The output should group the extracted entities by their type.
results = extractor.extract(text, schema)

# Print the extracted entities.
# The output should be a dictionary with an "entities" field.
print(results)

In [ ]:
# =========================
# Task 2: Named Entity Recognition on a real dataset
# Dataset: CoNLL-2003
# =========================

# This code demonstrates the NER capability of GLiNER2 on a small subset
# of the CoNLL-2003 validation set.
# We use only three common entity types:
# - person
# - organization
# - location
#
# To avoid the dataset loading issues in Colab, we first download the parquet
# file locally, then load it from the local path.

# -------------------------
# Step 0: Install required libraries
# -------------------------
!pip install -q datasets
!pip install -q -U datasets

from datasets import load_dataset
from gliner2 import GLiNER2

# -------------------------
# Step 1: Load the pretrained GLiNER2 model
# -------------------------
# "extractor" is the main model object used for entity extraction.
extractor = GLiNER2.from_pretrained("/content/gliner2-base-v1")
print("GLiNER2 model loaded successfully.")

# -------------------------
# Step 2: Download the CoNLL-2003 validation split locally
# -------------------------
# Dataset name: CoNLL-2003
# We download the validation parquet file to the Colab local storage
# so that we do not depend on remote dataset script loading.

dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")
# -------------------------
# Step 3: Load the local parquet file
# -------------------------

validation_data = dataset["validation"]
print(f"Number of validation samples: {len(validation_data)}")

# -------------------------
# Step 4: Select a small subset
# -------------------------
# We only use a small subset to keep the notebook simple and lightweight.
sample_size = 20
sample_data = validation_data.select(range(sample_size))
print(f"Number of evaluation samples: {len(sample_data)}")

# -------------------------
# Step 5: Define the entity types we want to evaluate
# -------------------------
# We only keep three common entity types for a simple project demonstration.
target_entity_types = ["person", "organization", "location"]

# -------------------------
# Step 6: Map CoNLL-2003 NER tags to our entity names
# -------------------------
# CoNLL-2003 uses token-level BIO tags:
# 0 = O
# 1 = B-PER, 2 = I-PER
# 3 = B-ORG, 4 = I-ORG
# 5 = B-LOC, 6 = I-LOC
# 7 = B-MISC, 8 = I-MISC
#
# We only use PER, ORG, and LOC in this project.
tag_to_entity = {
    1: "person", 2: "person",
    3: "organization", 4: "organization",
    5: "location", 6: "location"
}

# -------------------------
# Step 7: Helper function to reconstruct gold entities
# -------------------------
# This function converts token-level BIO tags into full entity strings.
def extract_gold_entities(tokens, ner_tags):
    entities = {
        "person": [],
        "organization": [],
        "location": []
    }

    current_tokens = []
    current_type = None

    for token, tag in zip(tokens, ner_tags):
        if tag == 0:
            # Outside any entity
            if current_tokens and current_type in entities:
                entities[current_type].append(" ".join(current_tokens))
            current_tokens = []
            current_type = None

        elif tag in [1, 3, 5]:
            # Beginning of a new entity
            if current_tokens and current_type in entities:
                entities[current_type].append(" ".join(current_tokens))
            current_tokens = [token]
            current_type = tag_to_entity[tag]

        elif tag in [2, 4, 6]:
            # Inside an existing entity
            entity_type = tag_to_entity[tag]
            if current_tokens and current_type == entity_type:
                current_tokens.append(token)
            else:
                # Handle irregular I-tags safely
                if current_tokens and current_type in entities:
                    entities[current_type].append(" ".join(current_tokens))
                current_tokens = [token]
                current_type = entity_type

    # Close the final entity if one remains open
    if current_tokens and current_type in entities:
        entities[current_type].append(" ".join(current_tokens))

    return entities

# -------------------------
# Step 8: Run NER on the subset
# -------------------------
# We compare GLiNER2 predictions with gold entities using exact string matches.
total_gold = 0
total_pred = 0
total_correct = 0

detailed_results = []

for example in sample_data:
    tokens = example["tokens"]
    ner_tags = example["ner_tags"]

    # Rebuild the sentence from tokens
    text = " ".join(tokens)

    # Extract gold entities from token-level annotations
    gold_entities = extract_gold_entities(tokens, ner_tags)

    # Use GLiNER2 to predict entities directly from text
    pred_results = extractor.extract_entities(
        text,
        target_entity_types
    )

    # Prediction format should contain an "entities" field
    if isinstance(pred_results, dict):
      pred_entities = pred_results.get("entities", pred_results)
    else:
      pred_entities = {}

    # Compare predictions with gold labels for each entity type
    for entity_type in target_entity_types:
        gold_list = gold_entities.get(entity_type, [])
        pred_list = pred_entities.get(entity_type, [])

        total_gold += len(gold_list)
        total_pred += len(pred_list)

        # Use set intersection to count exact matches
        gold_set = set(gold_list)
        pred_set = set(pred_list)
        total_correct += len(gold_set & pred_set)

    # Save a few sample examples for presentation
    detailed_results.append({
        "text": text,
        "gold": gold_entities,
        "pred": pred_entities
    })

# -------------------------
# Step 9: Compute simple precision / recall / F1
# -------------------------
precision = total_correct / total_pred if total_pred > 0 else 0
recall = total_correct / total_gold if total_gold > 0 else 0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

print("\nNER Results on CoNLL-2003 subset")
print(f"Total gold entities: {total_gold}")
print(f"Total predicted entities: {total_pred}")
print(f"Correct exact matches: {total_correct}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

# -------------------------
# Step 10: Print a few sample outputs
# -------------------------
print("\nSample NER predictions:")
for i in range(min(3, len(detailed_results))):
    print(f"\nExample {i+1}")
    print("Text:", detailed_results[i]["text"])
    print("Gold entities:", detailed_results[i]["gold"])
    print("Predicted entities:", detailed_results[i]["pred"])

# 3. Relation Extraction

In [ ]:
# This section demonstrates relation extraction.
# The goal is not only to find entities, but also to identify
# the relationship between them.

# Here, the schema defines two relation types:
# - "works_for"
# - "lives_in"
# The model will try to detect these semantic relations in the text.
schema = extractor.create_schema().relations([
    "works_for", "lives_in"
])

# This input sentence contains two relations:
# - John works for Apple Inc.
# - John lives in San Francisco.
text = "John works for Apple Inc. and lives in San Francisco."

# extractor.extract(...) performs relation extraction because the schema
# is defined with .relations(...).
# The output should return relation pairs in the form:
# (source entity, target entity)
results = extractor.extract(text, schema)

# Print the extracted relations.
# We expect something similar to:
# {
#   'relation_extraction': {
#       'works_for': [('John', 'Apple Inc.')],
#       'lives_in': [('John', 'San Francisco')]
#   }
# }
print(results)

In [ ]:
# =========================
# Task 3: Relation Extraction on a real dataset
# Dataset: CoNLL04
# =========================

# Install datasets library if needed.
!pip install -q datasets

from datasets import load_dataset
from gliner2 import GLiNER2

# -------------------------
# Step 1: Load the pretrained GLiNER2 model
# -------------------------
# We reuse the same local GLiNER2 model.
extractor = GLiNER2.from_pretrained("/content/gliner2-base-v1")
print("GLiNER2 model loaded successfully.")

# -------------------------
# Step 2: Load the CoNLL04 dataset
# -------------------------
# CoNLL04 is a standard relation extraction dataset.
# Each example contains:
# - tokens: tokenized sentence
# - entities: entity spans
# - relations: relation annotations between entities
dataset = load_dataset("DFKI-SLT/conll04")

# We use the validation split for a quick evaluation/demo.
validation_data = dataset["validation"]

# -------------------------
# Step 3: Select a small subset
# -------------------------
# To keep the notebook lightweight and fast, we only evaluate a small subset.
sample_size = 20
sample_data = validation_data.select(range(sample_size))
print(f"Number of evaluation samples: {len(sample_data)}")

# -------------------------
# Step 4: Define target relation types
# -------------------------
# We only keep three common and easier-to-explain relation types.
# This makes the project cleaner and easier to present.
target_relations = ["Work_For", "Live_In", "Located_In"]

# -------------------------
# Step 5: Helper function to rebuild entity text spans
# -------------------------
# In CoNLL04, each entity is stored as:
# - start: start token index
# - end: exclusive end token index
# - type: entity type
#
# This function converts one entity annotation into its text span.
def get_entity_text(tokens, entity):
    return " ".join(tokens[entity["start"]:entity["end"]])

# -------------------------
# Step 6: Helper function to extract gold relation tuples
# -------------------------
# Relations in the dataset are stored using:
# - head: index of the head entity in the entities list
# - tail: index of the tail entity in the entities list
# - type: relation type
#
# This function converts them into readable text tuples:
# (head_text, tail_text)
def extract_gold_relations(tokens, entities, relations, target_relations):
    gold = {rel_type: [] for rel_type in target_relations}

    for rel in relations:
        rel_type = rel["type"]
        if rel_type in target_relations:
            head_entity = entities[rel["head"]]
            tail_entity = entities[rel["tail"]]

            head_text = get_entity_text(tokens, head_entity)
            tail_text = get_entity_text(tokens, tail_entity)

            gold[rel_type].append((head_text, tail_text))

    return gold

# -------------------------
# Step 7: Run relation extraction and compare with gold labels
# -------------------------
# We count exact matches between predicted relation tuples and gold relation tuples.
total_gold = 0
total_pred = 0
total_correct = 0

detailed_results = []

for example in sample_data:
    tokens = example["tokens"]
    entities = example["entities"]
    relations = example["relations"]

    # Rebuild the sentence from tokens.
    text = " ".join(tokens)

    # Extract gold relation tuples from dataset annotations.
    gold_relations = extract_gold_relations(tokens, entities, relations, target_relations)

    # Use GLiNER2 to predict relations from raw text.
    pred_results = extractor.extract_relations(
        text,
        target_relations
    )

    # The model output should contain a dictionary under "relation_extraction".
    pred_relations = pred_results.get("relation_extraction", {})

    # Compare gold and predicted relations for each target type.
    for rel_type in target_relations:
        gold_list = gold_relations.get(rel_type, [])
        pred_list = pred_relations.get(rel_type, [])

        total_gold += len(gold_list)
        total_pred += len(pred_list)

        # Convert to sets so we can count exact tuple matches cleanly.
        gold_set = set(gold_list)
        pred_set = set(pred_list)
        total_correct += len(gold_set & pred_set)

    # Save a few detailed examples for presentation.
    detailed_results.append({
        "text": text,
        "gold": gold_relations,
        "pred": pred_relations
    })

# Step 8: Compute simple precision / recall / F1
# -------------------------
# These are simple exact-match scores on a small subset.
precision = total_correct / total_pred if total_pred > 0 else 0
recall = total_correct / total_gold if total_gold > 0 else 0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

print("\nRelation Extraction Results on CoNLL04 subset")
print(f"Total gold relations: {total_gold}")
print(f"Total predicted relations: {total_pred}")
print(f"Correct exact matches: {total_correct}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

# -------------------------
# Step 9: Print a few sample outputs
# -------------------------
# These examples are useful in the report and presentation.
print("\nSample relation extraction predictions:")
for i in range(3):
    print(f"\nExample {i+1}")
    print("Text:", detailed_results[i]["text"])
    print("Gold relations:", detailed_results[i]["gold"])
    print("Predicted relations:", detailed_results[i]["pred"])

# 4. Structured / JSON Extraction

In [ ]:
# This section demonstrates structured extraction.
# Unlike simple classification or entity extraction,
# the goal here is to return a structured JSON-like output.

# This is the input sentence.
# It contains product-related information:
# - product name
# - price
# - features
text = "The MacBook Pro costs $1999 and features M3 chip, 16GB RAM, and 512GB storage."

# extractor.extract_json(...) is used for structure-only extraction.
# We define one structure called "product".
# Inside "product", we ask the model to extract:
# - "name::str"     -> a single string value for the product name
# - "price"         -> price field (default behavior is allowed)
# - "features"      -> multiple feature values
#
# In other words, we are telling the model to organize the information
# into a JSON-like object instead of only returning labels.
results = extractor.extract_json(
    text,
    {
        "product": [
            "name::str",
            "price",
            "features"
        ]
    }
)

# Print the structured output.
# We expect something like:
# {
#   'product': [{
#       'name': 'MacBook Pro',
#       'price': ['$1999'],
#       'features': ['M3 chip', '16GB RAM', '512GB storage']
#   }]
# }
print(results)

In [ ]:
# =========================
# Task 4: Structured / JSON Extraction on a real dataset
# Dataset: GEM/e2e_nlg
# =========================

# This task uses the GEM/e2e_nlg dataset.
# GEM/e2e_nlg is a restaurant-domain data-to-text dataset.
# Each example contains a structured meaning representation (MR)
# and a natural language text describing the restaurant.
# Here, we reverse the direction:
# instead of generating text from structure, we extract structure from text.

!pip install -q datasets

from datasets import load_dataset
from gliner2 import GLiNER2
import re

# -------------------------
# Step 1: Load the pretrained GLiNER2 model
# -------------------------
# We reuse the same local GLiNER2 model.
extractor = GLiNER2.from_pretrained("/content/gliner2-base-v1")
print("GLiNER2 model loaded successfully.")

# -------------------------
# Step 2: Load the dataset
# -------------------------
# Dataset name: GEM/e2e_nlg
dataset = load_dataset("GEM/e2e_nlg", revision="convert/parquet")

# We use the validation split for a small demo/evaluation.
validation_data = dataset["validation"]

# -------------------------
# Step 3: Select a small subset
# -------------------------
# We only use a small subset to keep the notebook lightweight.
sample_size = 20
sample_data = validation_data.select(range(sample_size))
print(f"Number of evaluation samples: {len(sample_data)}")

# -------------------------
# Step 4: Helper functions
# -------------------------

# This function finds the text field and MR field in a robust way,
# because different dataset loaders sometimes use slightly different key names.
def get_text_and_mr(example):
    possible_text_keys = ["target", "text", "target_text"]
    possible_mr_keys = ["meaning_representation", "mr", "source"]

    text_value = None
    mr_value = None

    for key in possible_text_keys:
        if key in example:
            text_value = example[key]
            break

    for key in possible_mr_keys:
        if key in example:
            mr_value = example[key]
            break

    # Some versions store the text as a list of references.
    if isinstance(text_value, list):
        text_value = text_value[0]

    return text_value, mr_value

# This function parses a meaning representation string such as:
# name[The Eagle], eatType[coffee shop], food[French], area[riverside]
# into a Python dictionary.
def parse_mr(mr_string):
    pairs = re.findall(r'([^,\[]+)\[([^\]]*)\]', mr_string)
    parsed = {}
    for key, value in pairs:
        parsed[key.strip()] = value.strip()
    return parsed

# This function normalizes text for simple matching.
def normalize_text(s):
    if s is None:
        return ""
    return str(s).strip().lower()

# This function extracts a comparable predicted field value.
# GLiNER2 may return:
# - a string
# - a list
# - a dict with text/confidence
# We simplify it into plain text.
def simplify_pred_value(v):
    if isinstance(v, dict):
        return normalize_text(v.get("text", ""))
    elif isinstance(v, list):
        simplified_items = []
        for item in v:
            if isinstance(item, dict):
                simplified_items.append(normalize_text(item.get("text", "")))
            else:
                simplified_items.append(normalize_text(item))
        return simplified_items
    else:
        return normalize_text(v)

# -------------------------
# Step 5: Define the extraction schema
# -------------------------
# We define one restaurant structure.
# These fields are based on common E2E NLG restaurant attributes.
restaurant_schema = {
    "restaurant": [
        "name::str",
        "eatType::str",
        "food::str",
        "priceRange::str",
        "customer rating::str",
        "area::str",
        "familyFriendly::str",
        "near::str"
    ]
}

# -------------------------
# Step 6: Run structured extraction and compare with gold fields
# -------------------------
total_gold_fields = 0
total_correct_fields = 0

detailed_results = []

for example in sample_data:
    text, mr = get_text_and_mr(example)

    # Skip examples if fields are missing for any reason.
    if text is None or mr is None:
        continue

    # Parse the gold meaning representation.
    gold_fields = parse_mr(mr)

    # Run GLiNER2 structured extraction on the natural language text.
    pred = extractor.extract_json(text, restaurant_schema)

    # Safely get the first predicted restaurant object.
    pred_restaurants = pred.get("restaurant", [])
    pred_fields = pred_restaurants[0] if len(pred_restaurants) > 0 else {}

    # Compare only the fields that actually exist in the gold MR.
    correct_for_example = 0
    total_for_example = 0

    for field_name, gold_value in gold_fields.items():
        total_gold_fields += 1
        total_for_example += 1

        pred_value = pred_fields.get(field_name, "")

        gold_norm = normalize_text(gold_value)
        pred_simple = simplify_pred_value(pred_value)

        # If prediction is a list, count it as correct if gold value appears in the list.
        if isinstance(pred_simple, list):
            is_correct = gold_norm in pred_simple
        else:
            # For string fields, we allow containment in either direction
            # because extracted spans may be slightly longer or shorter.
            is_correct = (gold_norm == pred_simple) or (gold_norm in pred_simple) or (pred_simple in gold_norm)

        if is_correct:
            total_correct_fields += 1
            correct_for_example += 1

    # Save a few details for presentation.
    detailed_results.append({
        "text": text,
        "gold_fields": gold_fields,
        "pred_fields": pred_fields,
        "correct_fields": correct_for_example,
        "total_fields": total_for_example
    })

# -------------------------
# Step 7: Compute simple field-level accuracy
# -------------------------
field_accuracy = total_correct_fields / total_gold_fields if total_gold_fields > 0 else 0

print("\nStructured Extraction Results on GEM/e2e_nlg subset")
print(f"Total gold fields: {total_gold_fields}")
print(f"Correct extracted fields: {total_correct_fields}")
print(f"Field-level accuracy: {field_accuracy:.4f}")

# -------------------------
# Step 8: Print a few sample outputs
# -------------------------
print("\nSample structured extraction predictions:")
for i in range(min(3, len(detailed_results))):
    print(f"\nExample {i+1}")
    print("Text:", detailed_results[i]["text"])
    print("Gold fields:", detailed_results[i]["gold_fields"])
    print("Predicted fields:", detailed_results[i]["pred_fields"])
    print(f"Matched fields: {detailed_results[i]['correct_fields']}/{detailed_results[i]['total_fields']}")

# Conclusion and Future Direction

This project helped us understand how a single pretrained model can be used as a unified information extraction framework. Our implementation showed that GLiNER2 can support several NLP tasks through schema design, which makes it more flexible than systems built for only one task. It was also practical for our project because we could run the released model in Colab without reproducing a large training pipeline.

However, our work also had limitations. We implemented the released framework rather than training the full method from scratch, so our project should be seen as a simplified implementation. We also found that the output quality can depend on how clearly the schema and labels are defined. In future work, this method could be evaluated more systematically on benchmark datasets and compared more carefully against specialized task-specific models. It would also be useful to explore hierarchical structured extraction further, since the paper notes that this part still lacks strong zero-shot benchmark evaluation.

#References

[1]: Urchade Zaratiana, Nadi Tomeh, Pierre Holat, Thierry Charnois, GLiNER: Generalist Model for Named Entity Recognition using Bidirectional Transformer, arXiv, 2023

[2]: Urchade Zaratiana, Gil Pasternak, Oliver Boyd, George Hurn-Maloney, Ash Lewis, GLiNER2: An Efficient Multi-Task Information Extraction System with Schema-Driven Interface, Proceedings of the 2025 Conference on Empirical Methods in Natural Language Processing: System Demonstrations, 2025, pages 130–140